# Pumpkin Foliar Disease: 5-Fold Cross-Validation Segmentation Benchmark
> **Thesis:** Localization, Segmentation & Severity Estimation of Pumpkin Leaf Diseases  
> **Evaluation Protocol:** 5-Fold Stratified Cross-Validation on 100 Pilot Images (20 images/fold: 4 per class)  
> **Candidate Models Compared:**
> 1. **DualTaskUNet** — Unified single-stage dual-head U-Net (shared encoder-decoder, 31.04M params)
> 2. **Two-Stage Separate U-Net** — Independent Leaf U-Net + Lesion U-Net (reproducing [15], [16])
> 3. **LinkNet-ResNet18 (PDSNet-style)** — Lightweight encoder-decoder network (reproducing [15])
> 4. **DeepLabV3+** — Atrous spatial pyramid pooling architecture (reproducing [21])  
> **Metrics Reported:** Leaf Dice/IoU · Lesion Dice/IoU (Overall & Diseased-only) · Severity MAE (%) · Severity R² (Mean ± Std over 5 folds)


## Cell 0 — Setup Dependencies, Google Drive & Extract Pilot Dataset


In [2]:
# Install segmentation-models-pytorch for LinkNet and DeepLabV3+
!pip install -q segmentation-models-pytorch openpyxl

import os
import shutil
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
import segmentation_models_pytorch as smp

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PILOT_ZIP  = '/content/drive/MyDrive/pilot_100_images.zip'
PILOT_ROOT = '/content/pilot_100_images'

if not os.path.exists(PILOT_ROOT):
    print("Extracting pilot dataset archive...")
    !unzip -q "$PILOT_ZIP" -d /content/
    print("Extraction complete.")
else:
    print("Pilot folder already present.")

MASK_ROOT  = os.path.join(PILOT_ROOT, 'automated_masks')
LEAF_DIR   = os.path.join(MASK_ROOT, 'leaf')
LESION_DIR = os.path.join(MASK_ROOT, 'lesion')
OUT_DIR    = '/content/drive/MyDrive/pumpkin_severity_runs/cv5_segmentation_benchmark'
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | Output Directory: {OUT_DIR}")


Mounted at /content/drive
Extracting pilot dataset archive...
Extraction complete.
Device: cuda | Output Directory: /content/drive/MyDrive/pumpkin_severity_runs/cv5_segmentation_benchmark


## Cell 1 — Build 5-Fold Stratified Cross-Validation Folds


In [3]:
# Load all 100 images and match masks
classes_5 = ['BL', 'DM', 'HL', 'MD', 'PM']
class_full = {
    'BL': 'Bacterial Leaf Spot',
    'DM': 'Downy Mildew',
    'HL': 'Healthy Leaf',
    'MD': 'Mosaic Disease',
    'PM': 'Powdery Mildew'
}

records = []
for fname in sorted(os.listdir(PILOT_ROOT)):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    prefix = fname.split('_')[0]
    if prefix not in classes_5:
        continue

    img_path = os.path.join(PILOT_ROOT, fname)
    stem = os.path.splitext(fname)[0]
    leaf_p = os.path.join(LEAF_DIR, f"{stem}_leaf.png")
    lesion_p = os.path.join(LESION_DIR, f"{stem}_lesion.png")

    if os.path.exists(leaf_p) and os.path.exists(lesion_p):
        records.append({
            'filename': fname,
            'stem': stem,
            'class_code': prefix,
            'class_name': class_full[prefix],
            'img_path': img_path,
            'leaf_path': leaf_p,
            'lesion_path': lesion_p,
            'is_healthy': (prefix == 'HL')
        })

df_all = pd.DataFrame(records)
print(f"Total matched pilot samples: {len(df_all)}")
print(df_all['class_code'].value_counts())

# Stratified 5-Fold Split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
df_all['fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['class_code'])):
    df_all.loc[val_idx, 'fold'] = fold_idx

df_all.to_csv(os.path.join(OUT_DIR, 'cv5_splits_manifest.csv'), index=False)
print("5-Fold assignment complete. Distribution per fold:")
print(pd.crosstab(df_all['fold'], df_all['class_code']))


Total matched pilot samples: 80
class_code
DM    20
HL    20
MD    20
PM    20
Name: count, dtype: int64
5-Fold assignment complete. Distribution per fold:
class_code  DM  HL  MD  PM
fold                      
0            4   4   4   4
1            4   4   4   4
2            4   4   4   4
3            4   4   4   4
4            4   4   4   4


## Cell 2 — Dataset Class & Dual-Mask Loader


In [4]:
IMG_SIZE = (256, 256)

class PumpkinCVDataset(Dataset):
    def __init__(self, df, img_size=(256, 256), is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB').resize(self.img_size, Image.BILINEAR)
        leaf = Image.open(row['leaf_path']).convert('L').resize(self.img_size, Image.NEAREST)
        lesion = Image.open(row['lesion_path']).convert('L').resize(self.img_size, Image.NEAREST)

        # Horizontal flip augmentation during training
        if self.is_train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            leaf = leaf.transpose(Image.FLIP_LEFT_RIGHT)
            lesion = lesion.transpose(Image.FLIP_LEFT_RIGHT)

        img_arr = np.array(img, dtype=np.float32) / 255.0
        # ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_arr = (img_arr - mean) / std
        img_t = torch.tensor(img_arr).permute(2, 0, 1).float()

        leaf_t   = (torch.tensor(np.array(leaf) > 127).float()).unsqueeze(0)
        lesion_t = (torch.tensor(np.array(lesion) > 127).float()).unsqueeze(0)

        # Constrain lesion mask within leaf boundary
        lesion_t = lesion_t * leaf_t

        return {
            'img': img_t,
            'leaf': leaf_t,
            'lesion': lesion_t,
            'is_healthy': row['is_healthy'],
            'filename': row['filename'],
            'class_code': row['class_code']
        }

print("Dataset loader initialized.")


Dataset loader initialized.


## Cell 3 — Candidate Architectures Definition


In [5]:
# 1. DualTaskUNet (Single-Stage Dual-Head)
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class DualTaskUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        self.bottleneck = ConvBlock(512, 1024)

        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = ConvBlock(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        # Dual Bifurcated Heads
        self.head_leaf = nn.Conv2d(64, 1, 1)
        self.head_lesion = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head_leaf(d1), self.head_lesion(d1)

# Helper function to get candidate models
def get_model(model_name):
    if model_name == 'DualTaskUNet':
        return DualTaskUNet()
    elif model_name == 'Separate_UNet':
        # Single-target standard U-Net
        return smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1)
    elif model_name == 'LinkNet_ResNet18':
        # PDSNet-style (Zhao et al. [15])
        return smp.Linknet(encoder_name='resnet18', encoder_weights='imagenet', in_channels=3, classes=1)
    elif model_name == 'DeepLabV3Plus':
        # DeepLabV3+ with MobileNetV2 backbone (Gangwar et al. [21] family)
        return smp.DeepLabV3Plus(encoder_name='mobilenet_v2', encoder_weights='imagenet', in_channels=3, classes=1)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

print("Candidate architectures loaded.")


Candidate architectures loaded.


## Cell 4 — Loss Formulations & Evaluation Metrics


In [6]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2.0 * (probs * targets).sum(dim=(2, 3)) + self.smooth
        den = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth
        return 1.0 - (num / den).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        loss = self.alpha * ((1.0 - pt) ** self.gamma) * bce
        return loss.mean()

def compute_iou_dice(pred_bin, target_bin):
    inter = (pred_bin & target_bin).sum().item()
    union = (pred_bin | target_bin).sum().item()
    total = pred_bin.sum().item() + target_bin.sum().item()

    # Perfect match on empty ground truth
    if total == 0:
        return 1.0, 1.0
    iou = inter / union if union > 0 else 0.0
    dice = 2.0 * inter / total if total > 0 else 0.0
    return iou, dice

print("Losses and metrics compiled.")


Losses and metrics compiled.


## Cell 5 — 5-Fold Cross-Validation Execution Loop


In [7]:
EPOCHS = 20
BATCH_SIZE = 8
LR = 3e-4

# We benchmark DualTaskUNet and the 3 comparison architectures across all 5 folds
MODELS_TO_BENCHMARK = ['DualTaskUNet', 'Separate_UNet', 'LinkNet_ResNet18', 'DeepLabV3Plus']
cv_results = []

for m_name in MODELS_TO_BENCHMARK:
    print(f"\n=======================================================")
    print(f"BENCHMARKING: {m_name} (5-Fold Cross-Validation)")
    print(f"=======================================================")

    fold_metrics = []

    for fold in range(5):
        train_df = df_all[df_all['fold'] != fold]
        val_df   = df_all[df_all['fold'] == fold]

        train_loader = DataLoader(PumpkinCVDataset(train_df, is_train=True), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(PumpkinCVDataset(val_df, is_train=False), batch_size=BATCH_SIZE, shuffle=False)

        # Instantiate model(s)
        if m_name == 'DualTaskUNet':
            model = get_model(m_name).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
            bce_loss = nn.BCEWithLogitsLoss()
            dice_loss = DiceLoss()
            focal_loss = FocalLoss()

            # Train DualTaskUNet
            for ep in range(EPOCHS):
                model.train()
                for b in train_loader:
                    imgs = b['img'].to(device)
                    leaf_gt = b['leaf'].to(device)
                    les_gt  = b['lesion'].to(device)

                    l_pred, les_pred = model(imgs)
                    loss_leaf = bce_loss(l_pred, leaf_gt) + dice_loss(l_pred, leaf_gt)
                    loss_les  = focal_loss(les_pred, les_gt) + dice_loss(les_pred, les_gt)
                    loss = 1.0 * loss_leaf + 2.5 * loss_les

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

            # Validation on Fold
            model.eval()
            leaf_dices, leaf_ious = [], []
            les_dices_all, les_ious_all = [], []
            les_dices_dis = []
            sev_errors_all, sev_errors_dis = [], []
            gt_sevs, pred_sevs = [], []

            with torch.no_grad():
                for b in val_loader:
                    imgs = b['img'].to(device)
                    l_pred, les_pred = model(imgs)
                    l_bin = (torch.sigmoid(l_pred) > 0.5).cpu().numpy()
                    les_bin = (torch.sigmoid(les_pred) > 0.5).cpu().numpy()

                    for i in range(len(b['img'])):
                        cur_l_pred = l_bin[i, 0]
                        cur_les_pred = les_bin[i, 0] & cur_l_pred
                        cur_l_gt = (b['leaf'][i, 0].numpy() > 0.5)
                        cur_les_gt = (b['lesion'][i, 0].numpy() > 0.5)
                        is_hl = b['is_healthy'][i]

                        iou_l, dice_l = compute_iou_dice(cur_l_pred, cur_l_gt)
                        iou_les, dice_les = compute_iou_dice(cur_les_pred, cur_les_gt)

                        leaf_dices.append(dice_l); leaf_ious.append(iou_l)
                        les_dices_all.append(dice_les); les_ious_all.append(iou_les)

                        # Severity %
                        gt_sev = (cur_les_gt.sum() / max(1, cur_l_gt.sum())) * 100.0
                        pr_sev = (cur_les_pred.sum() / max(1, cur_l_pred.sum())) * 100.0
                        err = abs(pr_sev - gt_sev)

                        sev_errors_all.append(err)
                        gt_sevs.append(gt_sev); pred_sevs.append(pr_sev)

                        if not is_hl:
                            les_dices_dis.append(dice_les)
                            sev_errors_dis.append(err)

            # Fold summary
            fold_metrics.append({
                'leaf_dice': np.mean(leaf_dices),
                'leaf_iou': np.mean(leaf_ious),
                'lesion_dice_all': np.mean(les_dices_all),
                'lesion_dice_dis': np.mean(les_dices_dis),
                'sev_mae_all': np.mean(sev_errors_all),
                'sev_mae_dis': np.mean(sev_errors_dis)
            })
            print(f"  Fold {fold+1}/5 | LeafDice: {np.mean(leaf_dices):.4f} | LesDice(Dis): {np.mean(les_dices_dis):.4f} | SevMAE(Dis): {np.mean(sev_errors_dis):.2f}%")

        else:
            # Train single-target models (Separate U-Net, LinkNet, DeepLabV3+)
            # For each model, train a Leaf model and a Lesion model separately
            model_leaf = get_model(m_name).to(device)
            model_les  = get_model(m_name).to(device)

            opt_leaf = torch.optim.AdamW(model_leaf.parameters(), lr=LR, weight_decay=1e-4)
            opt_les  = torch.optim.AdamW(model_les.parameters(), lr=LR, weight_decay=1e-4)
            bce_loss = nn.BCEWithLogitsLoss()
            dice_loss = DiceLoss()
            focal_loss = FocalLoss()

            for ep in range(EPOCHS):
                model_leaf.train(); model_les.train()
                for b in train_loader:
                    imgs = b['img'].to(device)
                    leaf_gt = b['leaf'].to(device)
                    les_gt  = b['lesion'].to(device)

                    # Leaf optimization
                    l_pred = model_leaf(imgs)
                    l_loss = bce_loss(l_pred, leaf_gt) + dice_loss(l_pred, leaf_gt)
                    opt_leaf.zero_grad(); l_loss.backward(); opt_leaf.step()

                    # Lesion optimization
                    les_pred = model_les(imgs)
                    les_loss = focal_loss(les_pred, les_gt) + dice_loss(les_pred, les_gt)
                    opt_les.zero_grad(); les_loss.backward(); opt_les.step()

            # Validation on Fold
            model_leaf.eval(); model_les.eval()
            leaf_dices, leaf_ious = [], []
            les_dices_all, les_ious_all = [], []
            les_dices_dis = []
            sev_errors_all, sev_errors_dis = [], []

            with torch.no_grad():
                for b in val_loader:
                    imgs = b['img'].to(device)
                    l_pred = model_leaf(imgs)
                    les_pred = model_les(imgs)
                    l_bin = (torch.sigmoid(l_pred) > 0.5).cpu().numpy()
                    les_bin = (torch.sigmoid(les_pred) > 0.5).cpu().numpy()

                    for i in range(len(b['img'])):
                        cur_l_pred = l_bin[i, 0]
                        cur_les_pred = les_bin[i, 0] & cur_l_pred
                        cur_l_gt = (b['leaf'][i, 0].numpy() > 0.5)
                        cur_les_gt = (b['lesion'][i, 0].numpy() > 0.5)
                        is_hl = b['is_healthy'][i]

                        iou_l, dice_l = compute_iou_dice(cur_l_pred, cur_l_gt)
                        iou_les, dice_les = compute_iou_dice(cur_les_pred, cur_les_gt)

                        leaf_dices.append(dice_l); leaf_ious.append(iou_l)
                        les_dices_all.append(dice_les); les_ious_all.append(iou_les)

                        gt_sev = (cur_les_gt.sum() / max(1, cur_l_gt.sum())) * 100.0
                        pr_sev = (cur_les_pred.sum() / max(1, cur_l_pred.sum())) * 100.0
                        err = abs(pr_sev - gt_sev)

                        sev_errors_all.append(err)
                        if not is_hl:
                            les_dices_dis.append(dice_les)
                            sev_errors_dis.append(err)

            fold_metrics.append({
                'leaf_dice': np.mean(leaf_dices),
                'leaf_iou': np.mean(leaf_ious),
                'lesion_dice_all': np.mean(les_dices_all),
                'lesion_dice_dis': np.mean(les_dices_dis),
                'sev_mae_all': np.mean(sev_errors_all),
                'sev_mae_dis': np.mean(sev_errors_dis)
            })
            print(f"  Fold {fold+1}/5 | LeafDice: {np.mean(leaf_dices):.4f} | LesDice(Dis): {np.mean(les_dices_dis):.4f} | SevMAE(Dis): {np.mean(sev_errors_dis):.2f}%")

    # Aggregate over 5 folds (Mean ± Std)
    df_f = pd.DataFrame(fold_metrics)
    cv_results.append({
        'Model': m_name,
        'Leaf_Dice_mean': df_f['leaf_dice'].mean(),
        'Leaf_Dice_std': df_f['leaf_dice'].std(),
        'Leaf_IoU_mean': df_f['leaf_iou'].mean(),
        'Leaf_IoU_std': df_f['leaf_iou'].std(),
        'Lesion_Dice_Dis_mean': df_f['lesion_dice_dis'].mean(),
        'Lesion_Dice_Dis_std': df_f['lesion_dice_dis'].std(),
        'Sev_MAE_Dis_mean': df_f['sev_mae_dis'].mean(),
        'Sev_MAE_Dis_std': df_f['sev_mae_dis'].std(),
        'Sev_MAE_All_mean': df_f['sev_mae_all'].mean(),
        'Sev_MAE_All_std': df_f['sev_mae_all'].std()
    })

df_benchmark = pd.DataFrame(cv_results)
csv_out = os.path.join(OUT_DIR, 'cv5_benchmark_comparison.csv')
df_benchmark.to_csv(csv_out, index=False)
print("\n=======================================================")
print("FINAL 5-FOLD CROSS-VALIDATION BENCHMARK RESULTS")
print("=======================================================")
for idx, r in df_benchmark.iterrows():
    print(f"Model: {r['Model']:20s} | Leaf Dice: {r['Leaf_Dice_mean']:.4f}±{r['Leaf_Dice_std']:.4f} | Lesion Dice (Dis): {r['Lesion_Dice_Dis_mean']:.4f}±{r['Lesion_Dice_Dis_std']:.4f} | Sev MAE (Dis): {r['Sev_MAE_Dis_mean']:.2f}%±{r['Sev_MAE_Dis_std']:.2f}%")



BENCHMARKING: DualTaskUNet (5-Fold Cross-Validation)
  Fold 1/5 | LeafDice: 0.9642 | LesDice(Dis): 0.8528 | SevMAE(Dis): 2.77%
  Fold 2/5 | LeafDice: 0.9609 | LesDice(Dis): 0.8202 | SevMAE(Dis): 5.98%
  Fold 3/5 | LeafDice: 0.9362 | LesDice(Dis): 0.8340 | SevMAE(Dis): 9.34%
  Fold 4/5 | LeafDice: 0.9563 | LesDice(Dis): 0.8490 | SevMAE(Dis): 1.27%
  Fold 5/5 | LeafDice: 0.9682 | LesDice(Dis): 0.8703 | SevMAE(Dis): 2.74%

BENCHMARKING: Separate_UNet (5-Fold Cross-Validation)


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

  Fold 1/5 | LeafDice: 0.9526 | LesDice(Dis): 0.7765 | SevMAE(Dis): 2.82%
  Fold 2/5 | LeafDice: 0.9648 | LesDice(Dis): 0.7709 | SevMAE(Dis): 5.54%
  Fold 3/5 | LeafDice: 0.9519 | LesDice(Dis): 0.7792 | SevMAE(Dis): 3.48%
  Fold 4/5 | LeafDice: 0.9476 | LesDice(Dis): 0.7967 | SevMAE(Dis): 4.08%
  Fold 5/5 | LeafDice: 0.9579 | LesDice(Dis): 0.8193 | SevMAE(Dis): 5.25%

BENCHMARKING: LinkNet_ResNet18 (5-Fold Cross-Validation)


...

  Fold 1/5 | LeafDice: 0.9464 | LesDice(Dis): 0.7739 | SevMAE(Dis): 2.22%
  Fold 2/5 | LeafDice: 0.9593 | LesDice(Dis): 0.7321 | SevMAE(Dis): 4.11%
  Fold 3/5 | LeafDice: 0.9412 | LesDice(Dis): 0.7667 | SevMAE(Dis): 4.76%
  Fold 4/5 | LeafDice: 0.9335 | LesDice(Dis): 0.7937 | SevMAE(Dis): 4.25%
  Fold 5/5 | LeafDice: 0.9543 | LesDice(Dis): 0.7913 | SevMAE(Dis): 4.38%

BENCHMARKING: DeepLabV3Plus (5-Fold Cross-Validation)


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

  Fold 1/5 | LeafDice: 0.9322 | LesDice(Dis): 0.7454 | SevMAE(Dis): 2.89%
  Fold 2/5 | LeafDice: 0.9487 | LesDice(Dis): 0.7050 | SevMAE(Dis): 3.10%
  Fold 3/5 | LeafDice: 0.9334 | LesDice(Dis): 0.7313 | SevMAE(Dis): 4.48%
  Fold 4/5 | LeafDice: 0.9271 | LesDice(Dis): 0.7130 | SevMAE(Dis): 5.96%
  Fold 5/5 | LeafDice: 0.9437 | LesDice(Dis): 0.7670 | SevMAE(Dis): 3.44%

FINAL 5-FOLD CROSS-VALIDATION BENCHMARK RESULTS
Model: DualTaskUNet         | Leaf Dice: 0.9571±0.0125 | Lesion Dice (Dis): 0.8453±0.0190 | Sev MAE (Dis): 4.42%±3.24%
Model: Separate_UNet        | Leaf Dice: 0.9550±0.0066 | Lesion Dice (Dis): 0.7885±0.0197 | Sev MAE (Dis): 4.23%±1.16%
Model: LinkNet_ResNet18     | Leaf Dice: 0.9469±0.0103 | Lesion Dice (Dis): 0.7715±0.0248 | Sev MAE (Dis): 3.94%±0.99%
Model: DeepLabV3Plus        | Leaf Dice: 0.9370±0.0089 | Lesion Dice (Dis): 0.7323±0.0250 | Sev MAE (Dis): 3.97%±1.27%
